In [1]:
import os
import uuid

from dotenv import load_dotenv

import xarray as xr
import datetime

In [5]:
load_dotenv()

#OUT_ZARR = os.environ["POREALLAS_PARSED_ERA5_URI"]
START_YEAR = 1981
STOP_YEAR = 2025
TARGET_REGRID_URI = "/home/emily_zuetell/projects/poreallas/data/s51_hcm.nc"
UID = str(uuid.uuid4())
START_TIME = datetime.datetime.now(datetime.UTC).isoformat()

In [12]:
def open_regrid_target(uri: str) -> xr.Dataset:
    """Open/clean a dataset to use as a regridding target"""
    # Using the S51 seasonal monthly seasonal hindcast ensemble mean from copernicus as the target grid for our regrid...
    # Selecting so only have coords for latitude and longitude for regridding.
    target = xr.open_dataset(uri).isel(
        {"forecast_reference_time": 0, "forecastMonth": 0}, drop=True
    )
    return target

def avg_era5(base_file):
    era5_min = xr.open_dataset(f"{base_file}_min.nc")
    era5_max = xr.open_dataset(f"{base_file}_max.nc")

    era5 = (era5_min + era5_max)/2
    era5 = era5.rename({'valid_time': 'time'})
    return era5

In [6]:
regrid_target = open_regrid_target(TARGET_REGRID_URI)


In [13]:
era5 = avg_era5("/home/emily_zuetell/projects/poreallas/data/raw/era5_daily")

In [14]:
# Cannot have leap years in QDM bias adjustment so convert to a no-leapyear calendar.
era5 = era5.convert_calendar("noleap", dim="time")

regridder = xe.Regridder(era5, regrid_target, method="bilinear", periodic=True)
era5_regrid = regridder(era5)
era5_regrid.attrs |= era5.attrs

NameError: name 'xe' is not defined

In [15]:
import xesmf as xe

ModuleNotFoundError: No module named 'ESMF'